# Assignment 2: KL Divergence and Maximum Likelihood Estimation

This notebook covers the material from Lecture 3 on finding learning objectives:
- KL divergence: definition, properties, asymmetry
- Jensen's inequality
- Connecting KL minimisation to maximum likelihood
- MLE for the Gaussian distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)

---
## Part 1: KL Divergence

The **Kullback–Leibler divergence** from distribution $q$ to distribution $p$ is:

$$\text{KL}[p \| q] = \mathbb{E}_p\left[\log \frac{p(x)}{q(x)}\right] = \int p(x) \log \frac{p(x)}{q(x)} \, dx$$

Key properties:
1. $\text{KL}[p \| q] \geq 0$ (non-negative)
2. $\text{KL}[p \| q] = 0 \iff p = q$ (zero iff identical)
3. $\text{KL}[p \| q] \neq \text{KL}[q \| p]$ (asymmetric!)

### 1.1 Analytical KL between two Gaussians

For two univariate Gaussians $p = \mathcal{N}(\mu_1, \sigma_1^2)$ and $q = \mathcal{N}(\mu_2, \sigma_2^2)$, there is a closed form:

$$\text{KL}[p \| q] = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

In [ ]:
def kl_gaussians(mu1, sigma1, mu2, sigma2):
    """
    Closed-form KL[N(mu1,sigma1^2) || N(mu2,sigma2^2)].

    Returns:
        float, KL divergence
    """
    # TODO: implement the formula above
    raise NotImplementedError


# Sanity checks
print('KL[N(0,1) || N(0,1)] =', kl_gaussians(0, 1, 0, 1))  # expected: 0.0
print('KL[N(0,1) || N(1,1)] =', kl_gaussians(0, 1, 1, 1))  # expected: 0.5
print('KL[N(1,1) || N(0,1)] =', kl_gaussians(1, 1, 0, 1))  # expected: 0.5 (symmetric here!)
print('KL[N(0,1) || N(0,2)] =', kl_gaussians(0, 1, 0, 2))  # should differ from KL[N(0,2)||N(0,1)]

### 1.2 Asymmetry of KL divergence

A critical insight: $\text{KL}[p \| q]$ and $\text{KL}[q \| p]$ optimise for **different behaviours** when $p$ is the data distribution and $q$ is our model.

- **Forward KL** $\text{KL}[p_{\text{data}} \| q_\theta]$: model must cover all regions where $p_{\text{data}} > 0$ (mean-seeking, mode-covering)
- **Reverse KL** $\text{KL}[q_\theta \| p_{\text{data}}]$: model concentrates on modes of $p_{\text{data}}$ (mode-seeking, mass-covering)

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def kl_numerical(p_vals, q_vals, dx):
    """
    Numerically approximate KL[p || q] = sum_x p(x) * log(p(x)/q(x)) * dx.
    Assumes q(x) > 0 wherever p(x) > 0.

    Args:
        p_vals : np.ndarray, discretised p(x)
        q_vals : np.ndarray, discretised q(x)
        dx     : float, grid spacing

    Returns:
        float
    """
    # TODO: implement numerical KL
    # Avoid log(0) by masking where p_vals == 0
    raise NotImplementedError


# True distribution: bimodal mixture of two Gaussians
x = np.linspace(-6, 8, 1000)
dx = x[1] - x[0]
p_data = 0.5 * gaussian_pdf(x, -2, 0.8) + 0.5 * gaussian_pdf(x, 3, 0.8)

# Fit a single Gaussian q to minimise KL[p||q] by grid search over mu
mu_grid = np.linspace(-4, 6, 200)
forward_kls, reverse_kls = [], []

for mu_q in mu_grid:
    q = gaussian_pdf(x, mu_q, 1.0)
    q = q / (q.sum() * dx)  # renormalise
    forward_kls.append(kl_numerical(p_data, q, dx))
    reverse_kls.append(kl_numerical(q, p_data, dx))

mu_fwd = mu_grid[np.argmin(forward_kls)]
mu_rev = mu_grid[np.argmin(reverse_kls)]

plt.figure(figsize=(12, 5))
plt.plot(x, p_data, 'k-', lw=2, label='True p (bimodal)')
plt.plot(x, gaussian_pdf(x, mu_fwd, 1.0), 'b--', lw=2, label=f'Forward KL minimiser (mu={mu_fwd:.2f})')
plt.plot(x, gaussian_pdf(x, mu_rev, 1.0), 'r--', lw=2, label=f'Reverse KL minimiser (mu={mu_rev:.2f})')
plt.xlabel('x')
plt.title('Forward KL vs Reverse KL fitting of a unimodal q to a bimodal p')
plt.legend()
plt.show()

print(f'Forward KL minimised at mu = {mu_fwd:.2f}')
print(f'Reverse KL minimised at mu = {mu_rev:.2f}')

**Reflection**: In 2-3 sentences, explain why the forward and reverse KL minimisers end up in different places. Which behaviour do VAEs use, and why?

**Your answer**: TODO

---
## Part 2: Jensen's Inequality

**Jensen's inequality** states that for a **concave** function $f$:
$$f\left(\mathbb{E}[X]\right) \geq \mathbb{E}[f(X)]$$
For a **convex** function, the inequality flips.

The logarithm is concave, so: $\log \mathbb{E}[X] \geq \mathbb{E}[\log X]$.

This is the key step in the ELBO derivation (see Lecture 3).

In [ ]:
def verify_jensens(n_samples=10000):
    """
    Empirically verify Jensen's inequality for log:
        log(E[X]) >= E[log(X)]

    where X ~ Uniform(0.1, 3).
    """
    X = np.random.uniform(0.1, 3.0, n_samples)

    # TODO: compute log(E[X]) and E[log(X)]
    log_of_mean  = None  # TODO
    mean_of_log  = None  # TODO

    print(f'log(E[X])  = {log_of_mean:.4f}')
    print(f'E[log(X)]  = {mean_of_log:.4f}')
    print(f'Jensen gap = {log_of_mean - mean_of_log:.4f}  (should be >= 0)')

    # Visualise the gap
    x_plot = np.linspace(0.1, 3, 300)
    plt.figure(figsize=(7, 5))
    plt.plot(x_plot, np.log(x_plot), 'b-', lw=2, label='log(x)')
    E_X = X.mean()
    plt.axvline(E_X, color='gray', linestyle='--', label=f'E[X]={E_X:.2f}')
    plt.scatter([E_X], [log_of_mean], color='red', zorder=5, s=100, label='log(E[X])')
    plt.scatter([E_X], [mean_of_log], color='green', zorder=5, s=100, label='E[log(X)]')
    plt.legend()
    plt.title("Jensen's inequality: log is concave")
    plt.xlabel('x')
    plt.show()


verify_jensens()

### 2.1 Jensen's inequality in the ELBO derivation

The ELBO lower bound is derived by applying Jensen's inequality to the log-likelihood:

$$\ln p(x|\theta) = \ln \int p(x,z|\theta)\,dz = \ln \mathbb{E}_{q(z)}\left[\frac{p(x,z|\theta)}{q(z)}\right] \geq \mathbb{E}_{q(z)}\left[\ln \frac{p(x,z|\theta)}{q(z)}\right] = \mathcal{L}(q, \theta)$$

**Question**: In the derivation above, which function plays the role of $f$ in Jensen's inequality, and why is the inequality in the correct direction?

**Your answer**: TODO

---
## Part 3: Maximum Likelihood Estimation

We showed that minimising $\text{KL}[p_{\text{data}} \| p_\theta]$ is equivalent to maximising:

$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta \frac{1}{N} \sum_{n=1}^N \log p_\theta(x_n)$$

### 3.1 MLE for the Gaussian — derivation and implementation

In [ ]:
def mle_gaussian(X):
    """
    Compute the MLE estimates for a 1D Gaussian from data X.

    The MLE estimates are:
        mu_hat    = (1/N) * sum(x_i)
        sigma_hat = sqrt((1/N) * sum((x_i - mu_hat)^2))

    Note: this divides by N (biased), not N-1 (unbiased).

    Args:
        X : np.ndarray of shape (N,)

    Returns:
        mu_hat    : float
        sigma_hat : float
    """
    # TODO: implement MLE for Gaussian parameters
    raise NotImplementedError


# Test on data from a known distribution
true_mu, true_sigma = 3.0, 2.0
data = np.random.normal(true_mu, true_sigma, 500)

mu_hat, sigma_hat = mle_gaussian(data)
print(f'True:      mu={true_mu}, sigma={true_sigma}')
print(f'Estimated: mu={mu_hat:.3f}, sigma={sigma_hat:.3f}')

### 3.2 Log-likelihood curve

Visualise how the log-likelihood changes as a function of $\mu$, holding $\sigma$ fixed.

In [ ]:
def log_likelihood(X, mu, sigma):
    """
    Compute the average log-likelihood (1/N) * sum log N(x_n | mu, sigma^2).

    Args:
        X     : np.ndarray of shape (N,)
        mu    : float
        sigma : float

    Returns:
        float
    """
    # TODO: sum the log-PDF values over the dataset
    # Hint: log of Gaussian pdf = -0.5*log(2*pi*sigma^2) - (x-mu)^2 / (2*sigma^2)
    raise NotImplementedError


mu_range = np.linspace(-1, 7, 200)
ll_values = [log_likelihood(data, mu, true_sigma) for mu in mu_range]

plt.plot(mu_range, ll_values)
plt.axvline(mu_hat, color='red', linestyle='--', label=f'MLE mu={mu_hat:.2f}')
plt.axvline(true_mu, color='green', linestyle=':', label=f'True mu={true_mu}')
plt.xlabel('mu')
plt.ylabel('Average log-likelihood')
plt.title('Log-likelihood as a function of mu (sigma fixed)')
plt.legend()
plt.show()

### 3.3 Bias of MLE variance estimator

The MLE estimate $\hat{\sigma}^2 = \frac{1}{N}\sum(x_i - \hat{\mu})^2$ is **biased** — it systematically underestimates the true variance. The unbiased estimator divides by $N-1$.

In [ ]:
def compare_variance_estimators(true_sigma, n_repeats=1000, n_samples_list=None):
    """
    Compare MLE (biased, /N) and unbiased (/N-1) variance estimators
    across different sample sizes.
    """
    if n_samples_list is None:
        n_samples_list = [3, 5, 10, 20, 50, 100]

    mle_means, unbiased_means = [], []

    for N in n_samples_list:
        # TODO:
        # 1. Draw n_repeats datasets each of size N from N(0, true_sigma^2)
        # 2. For each dataset, compute MLE variance (divide by N) and unbiased (divide by N-1)
        # 3. Average across repeats
        mle_mean     = None  # TODO
        unbiased_mean = None  # TODO

        mle_means.append(mle_mean)
        unbiased_means.append(unbiased_mean)

    plt.plot(n_samples_list, mle_means, 'b-o', label='MLE (1/N)')
    plt.plot(n_samples_list, unbiased_means, 'r-o', label='Unbiased (1/(N-1))')
    plt.axhline(true_sigma**2, color='k', linestyle='--', label='True variance')
    plt.xlabel('N (sample size)')
    plt.ylabel('Expected variance estimate')
    plt.title('Bias of MLE variance estimator')
    plt.legend()
    plt.show()


compare_variance_estimators(true_sigma=2.0)

---
## Part 4: KL Minimisation = Log-Likelihood Maximisation

We showed in lecture that:
$$\text{KL}[p_{\text{data}} \| p_\theta] = -\mathbb{E}_{p_{\text{data}}}[\log p_\theta(x)] + \underbrace{\mathbb{E}_{p_{\text{data}}}[\log p_{\text{data}}(x)]}_{\text{constant w.r.t. }\theta}$$

So minimising KL is equivalent to maximising $\mathbb{E}_{p_{\text{data}}}[\log p_\theta(x)]$, which we approximate with $\frac{1}{N}\sum_n \log p_\theta(x_n)$.

In [ ]:
# Demonstrate numerically: minimise KL[p_data || q_theta] over mu
# where p_data is a fixed Gaussian N(2, 1) and q_theta = N(mu, 1)

true_mu_data = 2.0
N_data = 500
X_data = np.random.normal(true_mu_data, 1.0, N_data)

mu_candidates = np.linspace(-2, 6, 300)

# TODO:
# 1. For each mu in mu_candidates, compute:
#    (a) average_log_likelihood = mean log N(x | mu, 1) over X_data
#    (b) approx_kl = -average_log_likelihood  (up to a constant)
# 2. Find the mu that maximises log-likelihood / minimises KL
# 3. Plot both curves and mark the optimum

raise NotImplementedError

---
## Part 5: The Log-Sum-Exp Trick (preview for GMMs)

When computing $\log \sum_k \exp(a_k)$, naively exponentiating large values causes numerical overflow.

The **log-sum-exp trick** is:
$$\log \sum_k e^{a_k} = m + \log \sum_k e^{a_k - m}, \quad m = \max_k a_k$$

In [ ]:
def naive_log_sum_exp(a):
    """Numerically unstable version."""
    return np.log(np.sum(np.exp(a)))


def stable_log_sum_exp(a):
    """
    Numerically stable log-sum-exp.

    Args:
        a : np.ndarray of shape (K,)

    Returns:
        float
    """
    # TODO: implement the log-sum-exp trick described above
    raise NotImplementedError


# Test with large values that cause overflow in naive version
a_large = np.array([1000.0, 1001.0, 1002.0])
a_normal = np.array([1.0, 2.0, 3.0])

print('--- Normal values ---')
print('Naive:   ', naive_log_sum_exp(a_normal))
print('Stable:  ', stable_log_sum_exp(a_normal))
print('Scipy:   ', logsumexp(a_normal))

print('\n--- Large values (overflow test) ---')
print('Naive:   ', naive_log_sum_exp(a_large))   # will give inf
print('Stable:  ', stable_log_sum_exp(a_large))  # should work
print('Scipy:   ', logsumexp(a_large))

---
## Summary

In this notebook you:
- Derived and computed KL divergence analytically and numerically
- Observed the asymmetry of KL divergence and its effect on model fitting
- Verified Jensen's inequality empirically
- Implemented MLE for the Gaussian and proved the KL–MLE equivalence numerically
- Implemented the log-sum-exp trick for numerical stability

**Next**: Assignment 3 applies everything above to Gaussian Mixture Models and the EM algorithm.